<a href="https://colab.research.google.com/github/onizuka465/Pipeline-Def-Lab10/blob/main/Lab10Atividade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers bitsandbytes accelerate -q
!pip install -U bitsandbytes>=0.46.1 -q

In [4]:
!pip install flash-attn --no-build-isolation --find-links https://github.com/Dao-AILab/flash-attention/releases -q

ERROR: Operation cancelled by user


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

qlora_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

modelo_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(modelo_id)
modelo = AutoModelForCausalLM.from_pretrained(
    modelo_id,
    quantization_config=qlora_config,
    device_map="auto"
)

vram_usada = torch.cuda.memory_allocated() / 1024**2
print(f"VRAM utilizada após carregar o modelo: {vram_usada:.2f} MB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

VRAM utilizada após carregar o modelo: 784.50 MB


In [7]:
texto_medico = """
Paciente apresenta cefaleia pulsátil intensa com fotofobia e fonofobia.
Histórico de enxaqueca com aura. Pressão arterial 140/90 mmHg.
Diagnóstico diferencial inclui enxaqueca clássica e hipertensão arterial.
""" * 500  # repete para simular contexto massivo

inputs = tokenizer(texto_medico, return_tensors="pt", truncation=True, max_length=4096).to("cuda")

print(f"Total de tokens no contexto: {inputs['input_ids'].shape[1]}")

Total de tokens no contexto: 4096


 ## Geração sem KV Cache


In [11]:
import time

modelo.config.use_cache = False
torch.cuda.reset_peak_memory_stats()

inicio = time.time()

with torch.no_grad():
    output = modelo.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        use_cache=False
    )

fim = time.time()

tempo_sem_cache = fim - inicio
vram_sem_cache = torch.cuda.max_memory_allocated() / 1024**2

print(f"Tempo com KV Cache: {tempo_sem_cache:.2f} segundos")
print(f"Pico de VRAM com KV Cache: {vram_sem_cache:.2f} MB")


Tempo com KV Cache: 427.19 segundos
Pico de VRAM com KV Cache: 5905.49 MB


# Geração com KV Cache

In [12]:
import time

modelo.config.use_cache = True
torch.cuda.reset_peak_memory_stats()

inicio = time.time()

with torch.no_grad():
    output = modelo.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        use_cache=True
    )

fim = time.time()

tempo_com_cache = fim - inicio
vram_com_cache = torch.cuda.max_memory_allocated() / 1024**2

print(f"Tempo com KV Cache: {tempo_com_cache:.2f} segundos")
print(f"Pico de VRAM com KV Cache: {vram_com_cache:.2f} MB")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Tempo com KV Cache: 8.55 segundos
Pico de VRAM com KV Cache: 5755.32 MB


### Notas de Observação: Não consegui instalar a biblioteca do FlashAttention-2, o ambiente do colab não parece conseguir compilar a biblioteca.